In [2]:
import torch
import chess.pgn

from src.model import ResNet
from src.predict import get_uci_move
from src.dataset import create_value_csv_dataset, load_pgn 
from src.dataclass import ChessDataset

In [3]:
games = load_pgn("data/pgn/lichess_elite_2018-01.pgn")
create_value_csv_dataset(games, name="test")

In [5]:
val_dataset = ChessDataset("data/csv/test.csv")
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=256, shuffle=False)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet(filters=128, res_blocks=6)
checkpoint = torch.load(f"models/model_199.pth")
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

ResNet(
  (start_block): Sequential(
    (0): Conv2d(18, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
  )
  (res_tower): ModuleList(
    (0-5): 6 x ResBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (batch1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (batch2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
  )
  (policy_head): Sequential(
    (0): Conv2d(128, 2, kernel_size=(1, 1), stride=(1, 1))
    (1): BatchNorm2d(2, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Flatten(start_dim=1, end_dim=-1)
    (4): Linear(in_features=128, out_feat

In [10]:
game = chess.pgn.Game()
game.headers["Event"] = "Evaluation Game"
game.headers["Site"] = "Local"
game.headers["White"] = "Human"
game.headers["Black"] = "Model"

node = game

board = chess.Board()
while not board.is_game_over():
    move = get_uci_move(board, model, device)
    board.push(move)
    node = node.add_variation(move)
print(game)
print("\nGame over:", board.result())
with open("data/eval/version_a_1.pgn", "w") as pgn_file:
    print(game, file=pgn_file)

print("Game saved to game_output.pgn — upload it to Lichess to review!")

[Event "Evaluation Game"]
[Site "Local"]
[Date "????.??.??"]
[Round "?"]
[White "Human"]
[Black "Model"]
[Result "*"]

1. e4 c5 2. d4 cxd4 3. c3 d3 4. Bxd3 g6 5. Ne2 Bg7 6. O-O Nf6 7. e5 Ng4 8. Nf4 Nxe5 9. Be3 Nbc6 10. Bc2 O-O 11. Nd2 d6 12. Bd4 Nxd4 13. cxd4 Ng4 14. Re1 e5 15. dxe5 Nxf2 16. Kxf2 dxe5 17. Ne2 Qb6+ 18. Kf3 Qxb2 19. Rb1 Qxa2 20. Nb3 Bf6 21. Ng3 Bg7 22. Ne4 Bf5 23. g4 Bxe4+ 24. Rxe4 Rfd8 25. Rc1 a6 26. Qxd8+ Rxd8 27. h3 Rd7 28. Rb4 b5 29. Be4 g5 30. Rc6 Ra7 31. Nc5 a5 32. Rb3 b4 33. Rd3 Qa3 34. Nd7 Rxd7 35. Ke2 Qxd3+ 36. Bxd3 Ra7 37. Rc8+ Bf8 38. Ra8 Rxa8 39. Be4 b3 40. Bxa8 a4 41. Kd2 a3 42. Kd3 b2 43. Kc2 a2 44. Kxb2 a1=Q+ 45. Kxa1 Kg7 46. Kb1 Be7 47. Be4 Bd6 48. Bd5 f5 49. gxf5 Kf6 50. Be4 h5 51. Kc2 g4 52. hxg4 h4 53. Kd2 h3 54. Bf3 h2 55. Ke3 Bc7 56. Kf2 h1=Q 57. Bd5 e4 58. Be6 Bh2 59. Ke3 Be5 60. Kf2 Bh2 61. Ke3 Be5 62. Kf2 Bh2 63. Ke3 Be5 64. Kf2 Bh2 65. Ke3 Be5 66. Kf2 Qf3+ 67. Ke1 Qg2 68. Kd1 Bf4 69. Ke1 Qg1+ 70. Ke2 e3 71. Kf3 Qf1+ 72. Ke4 Qc4+ 73. Kf3 Qf1+ 74. 

In [12]:
import chess
import chess.pgn

import io

pgn = """[White "Stockfish"] [Black "CNN"]

1. e3 c5 2. Nf3 Nf6 3. Be2 Nc6 4. c4 g6 5. d4 cxd4 6. Nxd4 Bg7 
7. O-O d5 8. Nc3 O-O 9. cxd5 Nxd5 10. Qb3 Nxd4 11. exd4 Nxc3 
12. Qxc3 Bxd4 13. Qa3 Be6 14. Bg5 f6 15. Bh6 Rf7 16. Rac1 Bb6 
17. Bf3 Qd6 18. b4 Rd8 19. Bxb7 Qd5 20. h3 Qd4 21. Bc6 Bf5 
22. Be3 Qc4 23. Bxb6 axb6 24. Rxc4 h5 25. b5 Be6 26. Rcc1 Rd2 27. Rfe1 Bb3 
28. Kh1 Bc4 29. Rxc4 g5 30. Rg1 Kh7 31. h4 g4 32. Rcc1 Rxf2 
33. Qd3+ Rf5 34. Qa3 e5 35. Rc2 Rf4 36. Rcc1 Rb7 37. Bxb7 f5 
38. Bc8 e4 39. Qa7+ Kh6 40. Rc7 g3 41. Qxb6#"""

game = chess.pgn.read_game(io.StringIO(pgn))

board = game.board()
for move in game.mainline_moves():
    board.push(move)
    if board.fullmove_number == 27 and board.turn == chess.WHITE:
        print(board.fen())
        break

6k1/4pr2/1pB1bpp1/1P5p/8/Q6P/P2r1PP1/2R2RK1 w - - 3 27
